In [1]:
%env NUMBA_DEBUG_CACHE = 1

env: NUMBA_DEBUG_CACHE=1


In [2]:
from pathlib import Path
from tempfile import TemporaryFile

from numba.core.caching import CacheImpl, _CacheLocator

from pytensor import config
from pytensor.link.utils import COMPILED_SRC_FUNCTIONS


ENABLE = [True]


class NumbaPyTensorCacheLocator(_CacheLocator):
    def __init__(self, py_func, py_file, hash):
        # print(f"New locator {py_func=}, {py_file=}, {hash=}")
        self._py_func = py_func
        self._py_file = py_file
        self._hash = hash
        # src_hash = hash(pytensor_loader._module_sources[self._py_file])
        # self._hash = hash((src_hash, py_file, pytensor.__version__))

    def ensure_cache_path(self):
        # print("ensure_cache_path called")
        path = self.get_cache_path()
        path.mkdir(exist_ok=True)
        # Ensure the directory is writable by trying to write a temporary file
        TemporaryFile(dir=path).close()

    def get_cache_path(self):
        """
        Return the directory the function is cached in.
        """
        # print("get_cache_path called")
        return self._py_file

    def get_source_stamp(self):
        """
        Get a timestamp representing the source code's freshness.
        Can return any picklable Python object.
        """
        # return 0
        # print("get_source_stamp called")
        return self._hash

    def get_disambiguator(self):
        """
        Get a string disambiguator for this locator's function.
        It should allow disambiguating different but similarly-named functions.
        """
        # print("get_disambiguator called")
        return None

    @classmethod
    def from_function(cls, py_func, py_file):
        """
        Create a locator instance for the given function located in the given file.
        """
        # py_file = Path(py_file).parent
        # if py_file == (config.base_compiledir / "numba"):
        if ENABLE[0] and py_func in COMPILED_SRC_FUNCTIONS:
            # print(f"Applies to {py_file}")
            return cls(py_func, Path(py_file).parent, COMPILED_SRC_FUNCTIONS[py_func])


if CacheImpl._locator_classes[0] == NumbaPyTensorCacheLocator:
    CacheImpl._locator_classes.pop(0)
CacheImpl._locator_classes.insert(0, NumbaPyTensorCacheLocator)

In [3]:
config.base_compiledir

PosixPath('/home/ricardo/.pytensor')

In [4]:
import numpy as np

import pytensor
import pytensor.tensor as pt

In [5]:
x_test = np.random.normal(size=(5, 3))

In [6]:
x = pt.matrix("x")
y = pt.cos(x)

In [7]:
fn = pytensor.function([x], y, mode="NUMBA")
fn.trust_input = True
fn(x_test)
%timeit fn(x_test)

[cache] index loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Cos_cos_()_().store_core_outputs-None.py312.nbi'
[cache] data loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Cos_cos_()_().store_core_outputs-None.py312.1.nbc'
[cache] index saved to '/tmp/tmpc0p_48f_.numba_funcified_fgraph-None.py312.nbi'
[cache] data saved to '/tmp/tmpc0p_48f_.numba_funcified_fgraph-None.py312.1.nbc'
3.21 μs ± 297 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [8]:
fn(x_test).sum()

np.float64(7.297849553433626)

In [9]:
fn = pytensor.function([x], y, mode="NUMBA_VM")
fn.trust_input = True
fn(x_test)
%timeit fn(x_test)

[cache] index loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Cos_cos_()_().store_core_outputs-None.py312.nbi'
[cache] data loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Cos_cos_()_().store_core_outputs-None.py312.1.nbc'
3.4 μs ± 225 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [10]:
fn(x_test).sum()

np.float64(7.297849553433626)

In [11]:
fn = pytensor.function([x], y, mode="FAST_RUN")
fn.trust_input = True
%timeit fn(x_test)

1.59 μs ± 40.5 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [12]:
fn(x_test).sum()

np.float64(7.297849553433626)

In [57]:
import pytensor
import pytensor.tensor as pt


x = pt.matrix("x")
y = pt.exp(x)
# y = x.T

In [58]:
ENABLE[0] = True

In [59]:
%%timeit -r 1 -n 1
fn = pytensor.function([x], y, mode="NUMBA")
fn([[1, 2, 3]])

[cache] index loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Exp_exp_()_().store_core_outputs-None.py312.nbi'
[cache] data loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Exp_exp_()_().store_core_outputs-None.py312.1.nbc'
[cache] index saved to '/tmp/tmpfpqv0eil.numba_funcified_fgraph-None.py312.nbi'
[cache] data saved to '/tmp/tmpfpqv0eil.numba_funcified_fgraph-None.py312.1.nbc'
296 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [60]:
%%timeit -r 10 -n 10
fn = pytensor.function([x], y, mode="NUMBA")
# fn([[1, 2, 3]])

32.3 ms ± 670 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [61]:
%%timeit -r 10 -n 10
fn = pytensor.function([x], y, mode="NUMBA_VM")
# fn([[1, 2, 3]])

32.7 ms ± 1.15 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [18]:
%%timeit -r 1 -n 1
fn = pytensor.function([x], y, mode="NUMBA_VM")
fn([[1, 2, 3]])

[cache] index loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Exp_exp_()_().store_core_outputs-None.py312.nbi'
[cache] data loaded from '/home/ricardo/.pytensor/numba/store_core_outputs_Exp_exp_()_().store_core_outputs-None.py312.1.nbc'
187 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [24]:
%%timeit -r 1 -n 1
fn = pytensor.function([x], y, mode="FAST_RUN")
fn([[1, 2, 3]])

8.26 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [ ]:
COMPILED_SRC_FUNCTIONS

In [ ]:
%env NUMBA_DEBUG_CACHE = 1

In [ ]:
%env NUMBA_LLVM_PASS_TIMINGS = 1

In [ ]:
out = pytensor.function([x], y, mode="NUMBA")
out([[1, 2, 3]])

# md = out.vm.jit_fn.get_metadata(out.vm.jit_fn.signatures[0])
# print(md['llvm_pass_timings'])

In [ ]:
f = _DIMSHUFFLE_FNS[-1]
md = f.get_metadata(f.signatures[0])
print(md["llvm_pass_timings"])

In [ ]:
f.signatures

In [ ]:
out = pytensor.function([x], y, mode="NUMBA")
out([[1, 2, 3]])

# md = out.vm.jit_fn.get_metadata(out.vm.jit_fn.signatures[0])
# print(md['llvm_pass_timings'])

In [ ]:
_DIMSHUFFLE_FNS

In [ ]:
f = _DIMSHUFFLE_FNS[-1]
md = f.get_metadata(f.signatures[0])
print(md["llvm_pass_timings"])

In [ ]:
_DIMSHUFFLE_FNS